In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# --- Catppuccin dark theme colours ---
BG      = '#1e1e2e'
AX_BG   = '#181825'
TEXT    = '#cdd6f4'
BLUE    = '#89b4fa'
GREEN   = '#a6e3a1'
RED     = '#f38ba8'
ORANGE  = '#fab387'
PURPLE  = '#cba6f7'
YELLOW  = '#f9e2af'
SURFACE = '#313244'

# --- Table data ---
columns = ['Use Case', 'Domain', 'Best Model', 'AUC-ROC', 'F1', 'Type']
rows = [
    ['UC1', 'Synthetic Turbine', 'TGN (tuned)',        '0.816',      '0.600', 'Neural GNN'],
    ['UC1', 'Synthetic Turbine', 'IsolationForest',    '0.774',      '0.537', 'Anomaly Detection'],
    ['UC2', 'Oil Well (3W)',     'Random Forest',      '0.986',      '0.964', 'Tree Ensemble'],
    ['UC2', 'Oil Well (3W)',     'GradientBoosting',   '0.979',      '0.957', 'Tree Ensemble'],
    ['UC3', 'EPC Causal Delay',  'T-Logic R1+R2+R3',  '—',     '0.875', 'Symbolic Rules'],
    ['UC4', 'EPC Compliance',    'T-Logic R1+R2',      '—',     '1.000', 'Symbolic Rules'],
    ['UC4', 'EPC Compliance',    'Random Forest',      '0.933',      '0.525', 'Tree Ensemble'],
    ['UC4', 'EPC Compliance',    'TGN cert-aware',     '0.859',      '0.529', 'Neural GNN'],
    ['UC4', 'EPC Compliance',    'TNTComplEx',         '0.401 MRR',  '—', 'KG Embedding (link pred.)'],
]

# Row colour map by Type
type_colors = {
    'Neural GNN':                BLUE,
    'Anomaly Detection':         ORANGE,
    'Tree Ensemble':             GREEN,
    'Symbolic Rules':            RED,
    'KG Embedding (link pred.)': PURPLE,
}

fig, ax = plt.subplots(figsize=(16, 5.5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(AX_BG)
ax.axis('off')

col_widths = [0.07, 0.17, 0.20, 0.10, 0.07, 0.22]
x_starts = [0.01]
for w in col_widths[:-1]:
    x_starts.append(x_starts[-1] + w)

row_h = 0.088
header_y = 0.92

# Header row
for ci, (col, xs) in enumerate(zip(columns, x_starts)):
    ax.text(xs, header_y, col,
            transform=ax.transAxes,
            color=YELLOW, fontsize=10.5, fontweight='bold',
            va='top', ha='left')

# Separator line under header — use ax.plot with transAxes (axhline disallows transform kwarg)
sep_y = header_y - 0.035
ax.plot([0.01, 0.99], [sep_y, sep_y],
        color=SURFACE, linewidth=1.2, transform=ax.transAxes)

# Data rows
for ri, row in enumerate(rows):
    y = header_y - 0.055 - ri * row_h
    bg_col = '#24273a' if ri % 2 == 0 else AX_BG
    rect = mpatches.FancyBboxPatch(
        (0.005, y - 0.005), 0.99, row_h * 0.92,
        boxstyle='round,pad=0.005',
        linewidth=0, facecolor=bg_col,
        transform=ax.transAxes, zorder=0
    )
    ax.add_patch(rect)

    row_type = row[5]
    accent = type_colors.get(row_type, TEXT)

    for ci, (cell, xs) in enumerate(zip(row, x_starts)):
        color = accent if ci == 5 else TEXT
        fontw = 'bold' if ci == 0 else 'normal'
        ax.text(xs, y + row_h * 0.5, cell,
                transform=ax.transAxes,
                color=color, fontsize=9.5, fontweight=fontw,
                va='center', ha='left', zorder=1)

# Legend
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in type_colors.items()]
ax.legend(handles=legend_patches, loc='lower right',
          fontsize=8.5, framealpha=0.3,
          facecolor=SURFACE, edgecolor=TEXT,
          labelcolor=TEXT)

ax.set_title(
    'TKG Thesis — Results Across All Use Cases',
    color=TEXT, fontsize=14, fontweight='bold', pad=12,
    loc='left'
)

out_path = Path('../experiments/cross_uc_table.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out_path.resolve()}')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# --- Catppuccin dark theme colours ---
BG     = '#1e1e2e'
AX_BG  = '#181825'
TEXT   = '#cdd6f4'
BLUE   = '#89b4fa'
GREEN  = '#a6e3a1'
RED    = '#f38ba8'
ORANGE = '#fab387'
PURPLE = '#cba6f7'
YELLOW = '#f9e2af'
GRID   = '#313244'

# colour by model category
def model_color(name):
    n = name.lower()
    if 'tgn' in n:                              return BLUE
    if 'forest' in n or 'boosting' in n or 'lr' == n or n == 'rf': return GREEN
    if 'isolation' in n:                        return ORANGE
    if 'tntcomplex' in n:                       return PURPLE
    if 'symbolic' in n:                         return RED
    return TEXT

# --- Data ---
uc_data = {
    'UC1': [
        ('IsolationForest', 0.774),
        ('TGN (tuned)',     0.816),
    ],
    'UC2': [
        ('TGN baseline',      0.610),
        ('IsolationForest',   0.774),
        ('RandomForest',      0.986),
        ('GradientBoosting',  0.979),
    ],
    'UC4': [
        ('TGN_B',      0.767),
        ('TGN cert',   0.859),
        ('LR',         0.857),
        ('RF',         0.933),
    ],
}

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
fig.patch.set_facecolor(BG)

for ax, (uc, models) in zip(axes, uc_data.items()):
    ax.set_facecolor(AX_BG)
    names  = [m[0] for m in models]
    values = [m[1] for m in models]
    colors = [model_color(n) for n in names]
    x = np.arange(len(names))

    bars = ax.bar(x, values, color=colors, width=0.55, zorder=3,
                  edgecolor='#45475a', linewidth=0.6)

    # value labels on bars
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.008,
                f'{v:.3f}', ha='center', va='bottom',
                color=TEXT, fontsize=8.5, fontweight='bold')

    # good threshold dashed line
    ax.axhline(0.8, color=YELLOW, linestyle='--', linewidth=1.4,
               label='Good threshold (0.80)', zorder=4)

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha='right',
                       color=TEXT, fontsize=8.5)
    ax.set_ylim(0.5, 1.06)
    ax.set_title(uc, color=TEXT, fontsize=13, fontweight='bold', pad=8)
    if uc == 'UC1':
        ax.set_ylabel('AUC-ROC', color=TEXT, fontsize=10)
    ax.tick_params(colors=TEXT, which='both')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(GRID)
    ax.yaxis.grid(True, color=GRID, linestyle=':', linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    if uc == 'UC1':
        ax.legend(fontsize=8, framealpha=0.3,
                  facecolor='#313244', edgecolor=TEXT, labelcolor=TEXT)

# colour legend
legend_handles = [
    mpatches.Patch(color=BLUE,   label='Neural GNN'),
    mpatches.Patch(color=GREEN,  label='Tree / Linear'),
    mpatches.Patch(color=ORANGE, label='Anomaly Detection'),
    mpatches.Patch(color=PURPLE, label='KG Embedding'),
    mpatches.Patch(color=RED,    label='Symbolic'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=5,
           fontsize=9, framealpha=0.3,
           facecolor='#313244', edgecolor=TEXT, labelcolor=TEXT,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('AUC-ROC Comparison by Use Case',
             color=TEXT, fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()

out_path = Path('../experiments/cross_uc_auc.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out_path.resolve()}')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# --- Catppuccin dark theme colours ---
BG     = '#1e1e2e'
AX_BG  = '#181825'
TEXT   = '#cdd6f4'
BLUE   = '#89b4fa'
GREEN  = '#a6e3a1'
RED    = '#f38ba8'
ORANGE = '#fab387'
PURPLE = '#cba6f7'
YELLOW = '#f9e2af'
SURFACE= '#313244'
GRID   = '#45475a'

fig = plt.figure(figsize=(16, 8))
fig.patch.set_facecolor(BG)

# --- Left panel: T-Logic recall horizontal bar chart ---
ax_bar = fig.add_axes([0.05, 0.28, 0.42, 0.55])
ax_bar.set_facecolor(AX_BG)

tlogic_labels = [
    'UC3  R1+R2+R3\n(causal chains)',
    'UC4  R1+R2\n(compliance)',
]
f1_vals  = [0.875, 1.000]
bar_cols = [ORANGE, GREEN]
notes    = ['5/5 causal chains\ndetected', '449 violations\nzero missed']

y_pos = np.array([0.70, 0.25])
bar_h = 0.18
bars = ax_bar.barh(y_pos, f1_vals, height=bar_h, color=bar_cols,
                   edgecolor=GRID, linewidth=0.7, zorder=3)

for idx, (bar, v, note) in enumerate(zip(bars, f1_vals, notes)):
    ax_bar.text(v + 0.008, bar.get_y() + bar.get_height() / 2,
                f'F1 = {v:.3f}\n{note}',
                va='center', ha='left', color=TEXT, fontsize=9)
    ax_bar.text(-0.01, bar.get_y() + bar.get_height() / 2,
                tlogic_labels[idx],
                va='center', ha='right', color=TEXT, fontsize=9)

ax_bar.set_xlim(-0.01, 1.35)
ax_bar.set_ylim(0.02, 0.96)
ax_bar.axvline(1.0, color=YELLOW, linestyle='--', linewidth=1.3,
               label='Perfect F1', zorder=4)
ax_bar.set_xlabel('F1 Score', color=TEXT, fontsize=10)
ax_bar.set_yticks([])
ax_bar.tick_params(colors=TEXT)
ax_bar.spines[['top', 'right', 'left']].set_visible(False)
ax_bar.spines['bottom'].set_color(GRID)
ax_bar.xaxis.grid(True, color=GRID, linestyle=':', linewidth=0.7, zorder=0)
ax_bar.set_axisbelow(True)
ax_bar.legend(fontsize=8.5, framealpha=0.3,
              facecolor=SURFACE, edgecolor=TEXT, labelcolor=TEXT)
ax_bar.set_title('T-Logic Rule Recall Across Use Cases',
                 color=TEXT, fontsize=11, fontweight='bold', pad=8)

# --- Right panel: 2x2 thesis pillar text boxes ---
pillars = [
    {
        'pos': [0.555, 0.58, 0.19, 0.22],
        'title': '1. Bitemporal Modeling',
        'body': 'valid_time + tx_time\ncaptured in all 4 UCs',
        'color': BLUE,
    },
    {
        'pos': [0.775, 0.58, 0.19, 0.22],
        'title': '2. Symbolic Rules',
        'body': 'T-Logic rules\nR = 1.0 recall on both UCs',
        'color': RED,
    },
    {
        'pos': [0.555, 0.30, 0.19, 0.22],
        'title': '3. Neural Embedding',
        'body': 'TNTComplEx\nMRR = 0.401',
        'color': PURPLE,
    },
    {
        'pos': [0.775, 0.30, 0.19, 0.22],
        'title': '4. Domain Generalization',
        'body': '4 use cases\n3 data types',
        'color': GREEN,
    },
]

for p in pillars:
    ax_p = fig.add_axes(p['pos'])
    ax_p.set_facecolor(AX_BG)
    for spine in ax_p.spines.values():
        spine.set_edgecolor(p['color'])
        spine.set_linewidth(2)
    ax_p.set_xticks([])
    ax_p.set_yticks([])
    ax_p.text(0.5, 0.70, p['title'],
              transform=ax_p.transAxes,
              ha='center', va='center',
              color=p['color'], fontsize=9.5, fontweight='bold')
    ax_p.text(0.5, 0.28, p['body'],
              transform=ax_p.transAxes,
              ha='center', va='center',
              color=TEXT, fontsize=8.5, linespacing=1.5)

# Section label above pillars
fig.text(0.685, 0.88, 'Thesis Contributions',
         ha='center', va='bottom',
         color=YELLOW, fontsize=12, fontweight='bold')

# Overall title
fig.suptitle('Cross-Use-Case Contribution Summary',
             color=TEXT, fontsize=14, fontweight='bold', y=0.97)

out_path = Path('../experiments/cross_uc_contribution.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out_path.resolve()}')
